# Exercice 17 - Consumer Kafka avec Spark

## Objectifs
- Lire des messages Kafka avec Spark Structured Streaming
- Parser des messages JSON
- Traiter les donnees en streaming
- Ecrire les resultats dans MinIO

---

## 1. Architecture Consumer Spark

```
+------------------------------------------------------------------+
|               SPARK STRUCTURED STREAMING + KAFKA                 |
+------------------------------------------------------------------+
|                                                                  |
|   KAFKA                  SPARK                   DESTINATION    |
|                                                                  |
|  +--------+         +-------------+            +----------+     |
|  | Topic  |         |             |            |          |     |
|  |--------|  read   |  DataFrame  |   write    |  MinIO   |     |
|  | Part 0 |-------->|  Streaming  |----------->|  Parquet |     |
|  | Part 1 |         |             |            |          |     |
|  | Part 2 |         +------+------+            +----------+     |
|  +--------+                |                                    |
|                            v                                    |
|                     +------+------+            +----------+     |
|                     |             |            |          |     |
|                     | Aggregation |----------->|  Console |     |
|                     |             |            |          |     |
|                     +-------------+            +----------+     |
|                                                                  |
+------------------------------------------------------------------+

Modes de sortie :
- append   : Ajoute uniquement les nouvelles lignes
- complete : Reecrit toute la table (pour aggregations)
- update   : Met a jour les lignes modifiees
```

## 2. Configuration Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, to_timestamp, window,
    count, sum as spark_sum, avg, max as spark_max,
    explode, current_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, ArrayType, TimestampType
)

spark = SparkSession.builder \
    .appName("KafkaConsumer") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.565") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [2]:
# Configuration
KAFKA_BROKER = "broker:29092"
MINIO_BUCKET = "s3a://bronze"

print("Configuration prete")

Configuration prete


## 3. Lecture batch depuis Kafka

In [3]:
# Lire les messages existants (mode batch)
df_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load()

print(f"Messages lus: {df_kafka.count()}")
df_kafka.printSchema()

Messages lus: 66
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
# Structure des donnees Kafka
# - key: cle du message (binaire)
# - value: contenu du message (binaire)
# - topic: nom du topic
# - partition: numero de partition
# - offset: position dans la partition
# - timestamp: horodatage Kafka

df_kafka.select(
    col("key").cast("string").alias("key"),
    col("value").cast("string").alias("value"),
    "topic",
    "partition",
    "offset",
    "timestamp"
).show(5, truncate=50)

+--------+--------------------------------------------------+--------------+---------+------+-----------------------+
|     key|                                             value|         topic|partition|offset|              timestamp|
+--------+--------------------------------------------------+--------------+---------+------+-----------------------+
|CUST-002|{"order_id": 81767, "customer_id": "CUST-002", ...|commandes-json|        0|     0|2026-03-27 11:09:36.007|
|CUST-017|{"order_id": 39771, "customer_id": "CUST-017", ...|commandes-json|        0|     1|2026-03-27 11:09:36.007|
|CUST-002|{"order_id": 22800, "customer_id": "CUST-002", ...|commandes-json|        0|     2|2026-03-27 11:09:36.007|
|CUST-013|{"order_id": 58369, "customer_id": "CUST-013", ...|commandes-json|        0|     3|2026-03-27 11:09:36.007|
|CUST-017|{"order_id": 29958, "customer_id": "CUST-017", ...|commandes-json|        0|     4|2026-03-27 11:09:36.008|
+--------+----------------------------------------------

## 4. Parser les messages JSON

In [5]:
# Definir le schema des commandes
item_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("subtotal", DoubleType(), True)
])

commande_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("items", ArrayType(item_schema), True),
    StructField("total", DoubleType(), True),
    StructField("status", StringType(), True)
])

print("Schema defini")

Schema defini


In [6]:
# Parser le JSON
df_commandes = df_kafka \
    .select(
        col("key").cast("string").alias("customer_key"),
        from_json(col("value").cast("string"), commande_schema).alias("data"),
        "partition",
        "offset",
        col("timestamp").alias("kafka_timestamp")
    ) \
    .select(
        "customer_key",
        "data.*",
        "partition",
        "offset",
        "kafka_timestamp"
    )

df_commandes.printSchema()

root
 |-- customer_key: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: integer (nullable = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- quantity: integer (nullable = true)
 |    |    |-- unit_price: double (nullable = true)
 |    |    |-- subtotal: double (nullable = true)
 |-- total: double (nullable = true)
 |-- status: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)



In [7]:
# Afficher les commandes
df_commandes.select(
    "order_id",
    "customer_id",
    "total",
    "status",
    "partition"
).show(10)

+--------+-----------+-------+-------+---------+
|order_id|customer_id|  total| status|partition|
+--------+-----------+-------+-------+---------+
|   81767|   CUST-002| 209.96|created|        0|
|   39771|   CUST-017| 269.97|created|        0|
|   22800|   CUST-002| 299.96|created|        0|
|   58369|   CUST-013|2159.92|created|        0|
|   29958|   CUST-017| 939.92|created|        0|
|   35758|   CUST-017|  59.98|created|        0|
|   20040|   CUST-017| 159.98|created|        0|
|   40568|   CUST-020| 369.95|created|        0|
|   20667|   CUST-016|  29.99|created|        0|
|   13029|   CUST-016|1049.97|created|        0|
+--------+-----------+-------+-------+---------+
only showing top 10 rows


## 5. Analyser les items des commandes

In [8]:
# Exploser le tableau items
df_items = df_commandes \
    .select(
        "order_id",
        "customer_id",
        "timestamp",
        explode("items").alias("item")
    ) \
    .select(
        "order_id",
        "customer_id",
        "timestamp",
        col("item.product_id"),
        col("item.product_name"),
        col("item.quantity"),
        col("item.unit_price"),
        col("item.subtotal")
    )

df_items.show(10)

+--------+-----------+--------------------+----------+------------+--------+----------+--------+
|order_id|customer_id|           timestamp|product_id|product_name|quantity|unit_price|subtotal|
+--------+-----------+--------------------+----------+------------+--------+----------+--------+
|   81767|   CUST-002|2026-03-27T11:09:...|         6|      Webcam|       1|     89.99|   89.99|
|   81767|   CUST-002|2026-03-27T11:09:...|         7|     USB Hub|       3|     39.99|  119.97|
|   39771|   CUST-017|2026-03-27T11:09:...|         6|      Webcam|       2|     89.99|  179.98|
|   39771|   CUST-017|2026-03-27T11:09:...|         6|      Webcam|       1|     89.99|   89.99|
|   22800|   CUST-002|2026-03-27T11:09:...|         6|      Webcam|       1|     89.99|   89.99|
|   22800|   CUST-002|2026-03-27T11:09:...|         2|      Souris|       2|     29.99|   59.98|
|   22800|   CUST-002|2026-03-27T11:09:...|         5|      Casque|       1|    149.99|  149.99|
|   58369|   CUST-013|2026-03-

In [9]:
# Statistiques par produit
df_stats_produit = df_items.groupBy("product_name").agg(
    count("*").alias("nb_ventes"),
    spark_sum("quantity").alias("quantite_totale"),
    spark_sum("subtotal").alias("ca_total")
).orderBy(col("ca_total").desc())

df_stats_produit.show()

+------------+---------+---------------+------------------+
|product_name|nb_ventes|quantite_totale|          ca_total|
+------------+---------+---------------+------------------+
|      Laptop|       14|             27|26999.729999999996|
|       Ecran|       23|             46|16099.539999999999|
|         SSD|       20|             44|           5719.56|
|     Clavier|       30|             60|            4799.4|
|      Casque|       14|             28|           4199.72|
|      Webcam|       23|             42|           3779.58|
|     USB Hub|       19|             38|1519.6200000000001|
|      Souris|       20|             35|           1049.65|
+------------+---------+---------------+------------------+



## 6. Sauvegarder dans MinIO

In [10]:
# Sauvegarder les commandes en Parquet
df_commandes \
    .drop("items") \
    .write \
    .mode("overwrite") \
    .parquet(f"{MINIO_BUCKET}/kafka/commandes")

print("Commandes sauvegardees")

Commandes sauvegardees


In [11]:
# Sauvegarder les commandes en Parquet
df_commandes \
    .drop("items") \
    .write \
    .mode("overwrite") \
    .parquet(f"{MINIO_BUCKET}/kafka/commandes")

print("Commandes sauvegardees")

Commandes sauvegardees


In [12]:
# Sauvegarder les items
df_items \
    .write \
    .mode("overwrite") \
    .parquet(f"{MINIO_BUCKET}/kafka/items")

print("Items sauvegardes")

Items sauvegardes


## 7. Streaming en temps reel

In [13]:
# Lire en mode streaming
df_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "latest") \
    .load()

print("Stream configure")
print(f"Is streaming: {df_stream.isStreaming}")

Stream configure
Is streaming: True


In [14]:
# Parser le stream
df_stream_parsed = df_stream \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data"),
        "timestamp"
    ) \
    .select(
        "data.order_id",
        "data.customer_id",
        "data.total",
        "data.status",
        col("timestamp").alias("kafka_time")
    )

print("Stream parse")

Stream parse


In [28]:
# Ecrire dans la console (pour debug)
# Attention: Executez ce code puis envoyez des messages Kafka depuis un autre notebook

query = df_stream_parsed \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="5 seconds") \
    .start()

print("Stream demarre - envoyez des messages Kafka pour les voir")
print("Executez query.stop() pour arreter")

Stream demarre - envoyez des messages Kafka pour les voir
Executez query.stop() pour arreter


In [29]:
# Attendre quelques secondes puis arreter
import time
time.sleep(30)  # Attendre 30 secondes
query.stop()
print("Stream arrete")

Stream arrete


## 8. Aggregations en streaming

In [17]:
# Lire a nouveau le stream
df_stream2 = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "commandes-json") \
    .option("startingOffsets", "earliest") \
    .load()

# Parser
df_agg = df_stream2 \
    .select(
        from_json(col("value").cast("string"), commande_schema).alias("data"),
        "timestamp"
    ) \
    .select(
        "data.customer_id",
        "data.total",
        col("timestamp").alias("event_time")
    )

print("Stream prepare pour aggregation")

Stream prepare pour aggregation


In [18]:
# Aggregation par fenetre de temps
df_windowed = df_agg \
    .groupBy(
        window(col("event_time"), "1 minute"),
        "customer_id"
    ) \
    .agg(
        count("*").alias("nb_commandes"),
        spark_sum("total").alias("total_commandes")
    )

print("Aggregation definie")

Aggregation definie


In [19]:
# Ecrire les aggregations
query_agg = df_windowed \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="10 seconds") \
    .start()

print("Stream aggregation demarre")

Stream aggregation demarre


In [20]:
# Attendre puis arreter
time.sleep(20)
query_agg.stop()
print("Stream arrete")

Stream arrete


## 9. Lire les logs applicatifs

In [21]:
# Schema des logs
log_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("level", StringType(), True),
    StructField("module", StringType(), True),
    StructField("message", StringType(), True),
    StructField("request_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("duration_ms", IntegerType(), True)
])

# Lire les logs
df_logs = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", "logs-application") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(
        from_json(col("value").cast("string"), log_schema).alias("log")
    ) \
    .select("log.*")

print(f"Logs lus: {df_logs.count()}")
df_logs.show(10, truncate=False)

Logs lus: 15
+--------------------------+-------+--------+-------------------+----------+-------+-----------+
|timestamp                 |level  |module  |message            |request_id|user_id|duration_ms|
+--------------------------+-------+--------+-------------------+----------+-------+-----------+
|2026-03-27T11:20:42.011874|INFO   |api     |Rate limit exceeded|req-33962 |user-41|48         |
|2026-03-27T11:20:42.017365|INFO   |cache   |Cache updated      |req-33658 |user-30|199        |
|2026-03-27T11:20:42.019476|ERROR  |cache   |Cache cleared      |req-28506 |user-52|139        |
|2026-03-27T11:20:42.021051|INFO   |cache   |Cache updated      |req-34097 |user-43|171        |
|2026-03-27T11:20:42.022834|WARNING|cache   |Cache updated      |req-97213 |user-28|253        |
|2026-03-27T11:20:42.024901|WARNING|auth    |User logout        |req-27872 |user-78|34         |
|2026-03-27T11:20:42.026985|INFO   |payment |Payment initiated  |req-89625 |user-49|91         |
|2026-03-27T11:20

In [22]:
# Statistiques par niveau de log
df_logs.groupBy("level").count().show()

+-------+-----+
|  level|count|
+-------+-----+
|   INFO|    8|
|  ERROR|    4|
|WARNING|    3|
+-------+-----+



In [23]:
# Statistiques par module
df_logs.groupBy("module").agg(
    count("*").alias("nb_logs"),
    avg("duration_ms").alias("duree_moyenne_ms")
).orderBy(col("nb_logs").desc()).show()

+--------+-------+------------------+
|  module|nb_logs|  duree_moyenne_ms|
+--------+-------+------------------+
|   cache|      6| 278.1666666666667|
|     api|      3|             122.0|
|database|      3|190.66666666666666|
| payment|      2|             100.0|
|    auth|      1|              34.0|
+--------+-------+------------------+



In [24]:
# Fermer Spark
# spark.stop()

---

## Exercice

**Objectif** : Analyser les metriques Kafka

**Consigne** :
1. Lisez le topic "metrics" depuis Kafka
2. Parsez les messages JSON
3. Calculez les moyennes CPU et memoire par serveur
4. Identifiez les serveurs les plus charges

A vous de jouer :

In [52]:
from pyspark.sql.types import MapType
# TODO: Definir le schema des metriques
# Actual format:
# {
#   "timestamp": "2026-03-27T11:22:05.499049",
#   "host": "server-03",
#   "metrics": {
#     "cpu_percent": 24.55,
#     "memory_percent": 48.42,
#     "disk_percent": 38.45,
#     "network_in_mbps": 329.07,
#     "network_out_mbps": 192.21,
#     "requests_per_sec": 2679,
#     "response_time_ms": 55.65
#   }
# }

metrics_detail_schema = StructType([
    StructField("cpu_percent", DoubleType(), True),
    StructField("memory_percent", DoubleType(), True),
    StructField("disk_percent", DoubleType(), True),
    StructField("network_in_mbps", DoubleType(), True),
    StructField("network_out_mbps", DoubleType(), True),
    StructField("requests_per_sec", IntegerType(), True),
    StructField("response_time_ms", DoubleType(), True)
])

metric_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("host", StringType(), True),
    StructField("metrics", metrics_detail_schema, True)
])

In [53]:

# TODO: Lire le topic metrics

df_metrics = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("subscribe", "metrics")
    .option("startingOffsets", "earliest")
    .load()
    .select(from_json(col("value").cast("string"), metric_schema).alias("data"))
    .select(
        col("data.timestamp"),
        col("data.host"),
        col("data.metrics.cpu_percent"),
        col("data.metrics.memory_percent"),
        col("data.metrics.disk_percent"),
        col("data.metrics.network_in_mbps"),
        col("data.metrics.network_out_mbps"),
        col("data.metrics.requests_per_sec"),
        col("data.metrics.response_time_ms")
    )
)

print(f"Metrics lues: {df_metrics.count()}")
df_metrics.show(10, truncate=False)

Metrics lues: 10
+--------------------------+---------+-----------+--------------+------------+---------------+----------------+----------------+----------------+
|timestamp                 |host     |cpu_percent|memory_percent|disk_percent|network_in_mbps|network_out_mbps|requests_per_sec|response_time_ms|
+--------------------------+---------+-----------+--------------+------------+---------------+----------------+----------------+----------------+
|2026-03-27T11:22:05.499049|server-03|24.55      |48.42         |38.45       |329.07         |192.21          |2679            |55.65           |
|2026-03-27T11:22:05.501816|server-05|42.06      |35.88         |58.02       |121.61         |141.75          |4883            |62.95           |
|2026-03-27T11:22:05.502030|server-05|14.6       |60.05         |37.32       |298.34         |79.43           |3818            |79.58           |
|2026-03-27T11:22:05.502119|server-05|65.53      |79.13         |48.31       |349.01         |143.63       

In [54]:
# DEBUG: Verify metrics are being parsed correctly
print("=== PARSED METRICS ===")
df_metrics.printSchema()
df_metrics.show(5, truncate=False)

=== PARSED METRICS ===
root
 |-- timestamp: string (nullable = true)
 |-- host: string (nullable = true)
 |-- cpu_percent: double (nullable = true)
 |-- memory_percent: double (nullable = true)
 |-- disk_percent: double (nullable = true)
 |-- network_in_mbps: double (nullable = true)
 |-- network_out_mbps: double (nullable = true)
 |-- requests_per_sec: integer (nullable = true)
 |-- response_time_ms: double (nullable = true)

+--------------------------+---------+-----------+--------------+------------+---------------+----------------+----------------+----------------+
|timestamp                 |host     |cpu_percent|memory_percent|disk_percent|network_in_mbps|network_out_mbps|requests_per_sec|response_time_ms|
+--------------------------+---------+-----------+--------------+------------+---------------+----------------+----------------+----------------+
|2026-03-27T11:22:05.499049|server-03|24.55      |48.42         |38.45       |329.07         |192.21          |2679            |55.

In [57]:
# TODO: Calculer les statistiques par serveur

# CPU Statistics
df_cpu_stats = df_metrics.groupBy("host").agg(
    avg("cpu_percent").alias("avg_cpu"),
    spark_max("cpu_percent").alias("max_cpu")
).orderBy(col("avg_cpu").desc())

print("=== CPU Statistics par serveur ===")
df_cpu_stats.show(20, truncate=False)

# Memory Statistics
df_memory_stats = df_metrics.groupBy("host").agg(
    avg("memory_percent").alias("avg_memory"),
    spark_max("memory_percent").alias("max_memory")
).orderBy(col("avg_memory").desc())

print("\n=== Memory Statistics par serveur ===")
df_memory_stats.show(20, truncate=False)

# Combined - Servers most loaded (CPU + Memory average)
df_combined_load = df_metrics.groupBy("host").agg(
    avg("cpu_percent").alias("avg_cpu"),
    avg("memory_percent").alias("avg_memory")
).withColumn("avg_combined_load", (col("avg_cpu") + col("avg_memory")) / 2) \
.orderBy(col("avg_combined_load").desc())

print("\n=== Serveurs les plus charges (CPU+Memory) ===")
df_combined_load.show(10, truncate=False)

=== CPU Statistics par serveur ===
+---------+-------+-------+
|host     |avg_cpu|max_cpu|
+---------+-------+-------+
|server-01|71.685 |73.06  |
|server-05|52.955 |89.63  |
|server-03|40.68  |56.81  |
|server-04|35.93  |35.93  |
|server-02|18.47  |18.47  |
+---------+-------+-------+


=== Memory Statistics par serveur ===
+---------+----------+----------+
|host     |avg_memory|max_memory|
+---------+----------+----------+
|server-01|71.39     |79.25     |
|server-03|62.995    |77.57     |
|server-05|52.585    |79.13     |
|server-04|41.7      |41.7      |
|server-02|30.58     |30.58     |
+---------+----------+----------+


=== Serveurs les plus charges (CPU+Memory) ===
+---------+-------+----------+------------------+
|host     |avg_cpu|avg_memory|avg_combined_load |
+---------+-------+----------+------------------+
|server-01|71.685 |71.39     |71.5375           |
|server-05|52.955 |52.585    |52.769999999999996|
|server-03|40.68  |62.995    |51.8375           |
|server-04|35.93  

---

## Resume

Dans ce notebook, vous avez appris :
- Comment **lire des messages Kafka** avec Spark
- Comment **parser des messages JSON** avec un schema
- Comment utiliser le **mode batch** et le **mode streaming**
- Comment faire des **aggregations en streaming** avec des fenetres de temps
- Comment **sauvegarder les donnees** dans MinIO

### Prochaine etape
Dans le prochain notebook, nous approfondirons le streaming Spark avec des concepts avances.